In [ ]:
import import_ipynb
import pandas as pd
from tqdm.auto import tqdm
from scipy.optimize import minimize
import numpy as np
import math
from Data_Preprocessing import Data_Preprocessing
from Calculate_Returns import Calculate_Returns
from Technical_Indicators import Technical_Indicators

dp = Data_Preprocessing()

ti = Technical_Indicators()

In [ ]:
param_grid_values = {
    "ROC_direction": list(range(3,30,3)),
    "SMA_Trend_structure": [[5,10], [10,30], [10,50], [20,50]],
    "MACD_Trend_acceleration": list(range(3,30,3)),
    "Level_ZScore_Market_tension": list(range(3,30,3)),
    "RSI_Momentum_exhaustion": list(range(3,30,3)),
    "Ret_ZScore_Price_shock": list(range(3,30,3)),
    "Volatility_Market_risk": list(range(3,30,3)),
    "Volatility_Structure": list(range(3,30,3)),
    "Liquidity_Pressure": list(range(3,30,3))
}

In [ ]:
WEIGHTS = {
    "roc_weight": 1, # 1.5,
    "structure_weight": 1,# 1.5,
    "shock_weight": 1, #1.2,
    "rsi_weight": 1, #1.0,
    "macd_weight": 1, #1.2,
    "tension_weight": 1, #0.8,
    "risk_weight": 1, #1.0,
    "vol_structure_weight": 1, #0.8,
    "volume_weight": 1, #1.0
}

In [ ]:
PARAMS = {
    "ROC_direction": 20,
    "SMA_Trend_structure": [20,50],
    "MACD_Trend_acceleration": 9,
    "Level_ZScore_Market_tension": 20,
    "RSI_Momentum_exhaustion": 14,
    "Ret_ZScore_Price_shock": 7,
    "Volatility_Market_risk": 14,
    "Volatility_Structure": 20,
    "Liquidity_Pressure": 7
}

In [ ]:
class Param_Optimization():
    def generate_signals_voting_system(
        self,
        features,
        weights=WEIGHTS,
        structure_threshold=0,
        tension_threshold=1.5,
        shock_threshold=2,
        risk_quantile=0.75,
        risk_window=200
    ):
        signals = pd.DataFrame(index=features.index)
        signals["signal"] = 0

        if "ROC_direction" in features:
            # # === 1️⃣ MOMENTUM (ROC) ===
            roc_threshold = features["ROC_direction"].rolling(200).std()
            signals.loc[features["ROC_direction"] > roc_threshold, "signal"] += weights["roc_weight"]
            signals.loc[features["ROC_direction"] < -roc_threshold, "signal"] -= weights["roc_weight"]
            
        if "SMA_Trend_structure" in features:
            # === 2️⃣ TREND STRUCTURE ===
            signals.loc[features["SMA_Trend_structure"] > structure_threshold, "signal"] += weights["structure_weight"]
            signals.loc[features["SMA_Trend_structure"] < -structure_threshold, "signal"] -= weights["structure_weight"]
            
        if "Level_ZScore_Market_tension" in features:
            # === 3️⃣ MARKET TENSION (mean reversion logic) ===
            signals.loc[features["Level_ZScore_Market_tension"] < -tension_threshold, "signal"] += weights["tension_weight"]  # oversold
            signals.loc[features["Level_ZScore_Market_tension"] > tension_threshold, "signal"] -= weights["tension_weight"]  # overbought

        if "SMA_Trend_structure" in features and "Ret_ZScore_Price_shock" in features:
            # === 4️⃣ PRICE SHOCK (context-aware) ===
            uptrend = features["SMA_Trend_structure"] > 0
            downtrend = features["SMA_Trend_structure"] < 0

            # breakout continuation
            signals.loc[(features["Ret_ZScore_Price_shock"] > shock_threshold) & uptrend, "signal"] += weights["shock_weight"]
            signals.loc[(features["Ret_ZScore_Price_shock"] < -shock_threshold) & downtrend, "signal"] -= weights["shock_weight"]

            # panic reversal
            signals.loc[(features["Ret_ZScore_Price_shock"] < -shock_threshold) & uptrend, "signal"] += weights["shock_weight"]
            signals.loc[(features["Ret_ZScore_Price_shock"] > shock_threshold) & downtrend, "signal"] -= weights["shock_weight"]

        if "Volatility_Market_risk" in features:
            # === 5️⃣ RISK REGIME (NO LEAKAGE) ===
            uptrend = features["SMA_Trend_structure"] > 0
            downtrend = features["SMA_Trend_structure"] < 0
            rolling_risk_thresh = (
                features["Volatility_Market_risk"]
                .rolling(risk_window)
                .quantile(risk_quantile)
            )
            
            signals.loc[(features["Volatility_Market_risk"] > rolling_risk_thresh) & uptrend, "signal"] -= weights["risk_weight"]
            signals.loc[(features["Volatility_Market_risk"] > rolling_risk_thresh) & downtrend, "signal"] += weights["risk_weight"]
        
        if "RSI_Momentum_exhaustion" in features and "SMA_Trend_structure" in features:
            # === 6️⃣ RSI (momentum exhaustion) ===
            rsi = features["RSI_Momentum_exhaustion"]

            uptrend = features["SMA_Trend_structure"] > 0
            downtrend = features["SMA_Trend_structure"] < 0

            # oversold w trendzie wzrostowym = BUY
            signals.loc[(rsi < 30) & uptrend, "signal"] += weights["rsi_weight"]

            # overbought w trendzie spadkowym = SELL
            signals.loc[(rsi > 70) & downtrend, "signal"] -= weights["rsi_weight"]

            # ekstremalne stany w bokach rynku
            sideways = features["SMA_Trend_structure"].abs() < 0.001
            signals.loc[(rsi < 25) & sideways, "signal"] += weights["rsi_weight"]
            signals.loc[(rsi > 75) & sideways, "signal"] -= weights["rsi_weight"]
        
        if "MACD_Trend_acceleration" in features and "SMA_Trend_structure" in features:
            # === 7 MACD Acceleration ===
            uptrend = features["SMA_Trend_structure"] > 0
            downtrend = features["SMA_Trend_structure"] < 0
            signals.loc[(features["MACD_Trend_acceleration"] > 0) & uptrend, "signal"] += weights["macd_weight"]
            signals.loc[(features["MACD_Trend_acceleration"] < 0) & downtrend, "signal"] -= weights["macd_weight"]  
        
        if "Volatility_Structure" in features:
            # === 8 Volatility structure ===
            squeeze = features["Volatility_Structure"] < features["Volatility_Structure"].rolling(100).quantile(0.2)
            expansion = features["Volatility_Structure"] > features["Volatility_Structure"].rolling(100).quantile(0.8)
            signals.loc[squeeze & uptrend, "signal"] += weights["vol_structure_weight"]
            signals.loc[squeeze & downtrend, "signal"] -= weights["vol_structure_weight"]
            signals.loc[expansion, "signal"] -= 1  # chaos → ostrożność
        
        if "Liquidity_Pressure" in features:
            # === 9 Volume Confirmation ===
            signals.loc[(features["Liquidity_Pressure"] > 1) & uptrend, "signal"] += weights["volume_weight"]
            signals.loc[(features["Liquidity_Pressure"] > 1) & downtrend, "signal"] -= weights["volume_weight"]
        
        return signals
    
    
    def simple_simulate(self, params=PARAMS):
        stock_aapl = dp.load_stock_data_from_file("AAPL")
        stock_aapl["pct_change"] = stock_aapl["close"].pct_change()
        features = ti.build_market_features(stock_aapl["close"], stock_aapl["volume"], params)
        votes = self.generate_signals_voting_system(features)
        df = dp.merge_data(stock_aapl, votes, False)
        df['signal'] = df['signal'].apply(lambda x: 1 if x >= 1 else -1 if x <= -1 else 0)
        df["return"] = df["pct_change"] * df['signal'].shift(1)
        df = df.dropna()
        std = df["return"].std()
        sharpe = (df["return"].mean() / std) * math.sqrt(252)
        win_rate = (len(df[df["return"] > 0])/len(df)) * 100
        cumulative_return = (1 + df["return"]).prod() - 1
        return sharpe, win_rate, cumulative_return
    
    
    def advanced_simulate(self, params):
        initial_cash = 10000
        fee = 0.02
        stock_aapl = dp.load_stock_data_from_file("AAPL")
        stock_aapl["pct_change"] = stock_aapl["close"].pct_change()
        features = ti.build_market_features(stock_aapl["close"], stock_aapl["volume"], params)
        votes = self.generate_signals_voting_system(features)
        df = dp.merge_data(stock_aapl, votes, False)
        df['signal'] = df['signal'].apply(lambda x: 1 if x >= 1 else -1 if x <= -1 else 0)
        df['tp_stop'] = 0
        df['sl_stop'] = 0
        cr = Calculate_Returns(
            initial_cash=initial_cash,
            fee=fee,
            prices=df['close'],
            signals=df['signal'],
            tp_stop=df['tp_stop'],
            sl_stop=df['sl_stop']
        )
        cr.from_signals()
        returns_array = np.array([float(r) for r in cr.returns])
        win_rate = (len(returns_array[returns_array > 0])/len(returns_array)) * 100
        return cr.sharpe(), win_rate  # Minimize the negative Sharpe ratio to maximize it
    
    def optimize(self, data, num_starts=10, initial_cash=100000, fee=0.006, freq='3MS', param_grid=param_grid_values):
        optimized_params = []
        intervals = pd.date_range(start=data.index.min(), end=data.index.max(), freq=freq)

        for start, end in zip(intervals[:-1], intervals[1:]):
            best_sharpe_ratio, best_net_profit, best_params = -np.inf, -np.inf, None
            interval_df = data.loc[start:end]

            param_space = [
                param_grid_values["ROC_direction"],
                param_grid_values["SMA_Trend_structure"],
                param_grid_values["MACD_Trend_acceleration"],
                param_grid_values["Level_ZScore_Market_tension"],
                param_grid_values["RSI_Momentum_exhaustion"],
                param_grid_values["Ret_ZScore_Price_shock"],
                param_grid_values["Volatility_Market_risk"],
                param_grid_values["Volatility_Structure"],
                param_grid_values["Liquidity_Pressure"]
            ]

            def bounds_to_params(x):
                """Converts a list of indices to actual parameter values"""
                return {key: space[int(idx)] for key, space, idx in zip(param_grid.keys(), param_space, x)}

            def objective_wrapper(x):
                """Wrapper for the optimization objective: calculates negative Sharpe ratio"""
                params = bounds_to_params(x)
                
                features = ti.build_market_features(interval_df["close"], interval_df["volume"], params)
                votes = self.generate_signals_voting_system(features)
                merged = dp.merge_data(interval_df, votes, False)
                merged['signal'] = merged['signal'].apply(lambda x: 2 if x > 1 else 1 if x < -1 else 0)
                merged['tp_stop'] = 0
                merged['sl_stop'] = 0
                cr = Calculate_Returns(
                    initial_cash=initial_cash,
                    fee=fee,
                    prices=merged['close'],
                    signals=merged['signal'],
                    tp_stop=merged['tp_stop'],
                    sl_stop=merged['sl_stop']
                )
                cr.from_signals()
                return -cr.sharpe()  # Minimize the negative Sharpe ratio to maximize it
            
            # def objective_wrapper(x):
            #     params = bounds_to_params(x)
            #     df = interval_df.copy()
            #     df["pct_change"] = df["close"].pct_change()
            #     features = ti.build_market_features(df["close"], df["volume"], params)
            #     votes = self.generate_signals_voting_system(features)
            #     df = dp.merge_data(df, votes, False)
            #     df['signal'] = df['signal'].apply(lambda x: 1 if x >= 1 else -1 if x <= -1 else 0)
            #     df["returns"] = df["signal"].shift(1) * df["pct_change"]
            #     df = df.dropna(subset=["returns"])
            #     if df["returns"].dropna().empty:
            #         return 1e6
            #     std = df["returns"].std()
            #     if std == 0 or np.isnan(std):
            #         return 1e6
            #     sharpe = (df["returns"].mean() / std) * math.sqrt(365)
            #     return -sharpe

            # Define the bounds for each parameter based on the parameter space
            bounds = [(0, len(space) - 1) for space in param_space]

            for _ in tqdm(range(num_starts), desc="Optimizing"):
                initial_guess = [np.random.randint(len(space)) for space in param_space]

                result = minimize(objective_wrapper, initial_guess, method='SLSQP', bounds=bounds)

                if result.success and -result.fun > best_sharpe_ratio:
                    best_sharpe_ratio = -result.fun
                    best_params = bounds_to_params(result.x)

            optimized_params.append({
                'start': start,
                'end': end,
                'params': best_params,
                'sharpe_ratio': best_sharpe_ratio,
            })

        return pd.DataFrame(optimized_params)
